# Train gesture classifier
Loads `data/annotations/<label>.npy` files, normalizes landmarks, trains a small MLP, saves `models/gesture_classifier.pth`.

In [ ]:
import sys
sys.path.append('..')

import glob
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.components.classifier import GestureNet
from src.utils.landmarks import flatten, normalize

In [ ]:
ANNOTATIONS_DIR = '../data/annotations'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

labels = sorted(os.path.splitext(os.path.basename(p))[0]
                 for p in glob.glob(f'{ANNOTATIONS_DIR}/*.npy'))
print('labels:', labels)

X, y = [], []
for i, label in enumerate(labels):
    samples = np.load(f'{ANNOTATIONS_DIR}/{label}.npy')
    for lm in samples:
        X.append(flatten(normalize(lm)))
        y.append(i)

X = np.stack(X).astype(np.float32)
y = np.array(y, dtype=np.int64)
np.save(f'{PROCESSED_DIR}/X.npy', X)
np.save(f'{PROCESSED_DIR}/y.npy', y)
print(X.shape, y.shape)

In [ ]:
dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = GestureNet(num_classes=len(labels))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 30
for epoch in range(EPOCHS):
    total_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f'epoch {epoch}: loss={total_loss / len(loader):.4f}')

In [ ]:
os.makedirs('../models', exist_ok=True)
torch.save({'labels': labels, 'state_dict': model.state_dict()}, '../models/gesture_classifier.pth')
print('saved ../models/gesture_classifier.pth')